# Actividad — Bagging y Random Forests
**Aprendizaje Supervisado · Sesión 13 · Bagging y bosques aleatorios**

**Qué haremos.** En la presentación vimos cómo combinar modelos **en paralelo**: bagging (remuestreo bootstrap + promedio) y Random Forest (bagging + aleatoriedad extra en los cortes). Aquí los probamos sobre un conjunto de datos sintético y comparamos su comportamiento.

Trabaja de forma **individual**. Donde veas `...` con un comentario `# ✏️`, completa esa línea; donde veas **✏️ Pregunta**, responde en la celda de markdown siguiente.

## Parte 0 — Los datos

Generamos un conjunto sintético de clasificación binaria: 800 observaciones, 10 variables numéricas, de las cuales solo 5 son realmente informativas (las otras 5 son ruido). Esto nos deja comparar, con la "respuesta correcta" en mente, qué tan bien cada modelo identifica las variables que importan.

In [ ]:
import warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                               AdaBoostClassifier, GradientBoostingClassifier)
warnings.filterwarnings("ignore")

X, y = make_classification(n_samples=800, n_features=10, n_informative=5, n_redundant=2,
                            n_clusters_per_class=2, class_sep=1.1, random_state=0)
cols = [f"x{i}" for i in range(10)]
X = pd.DataFrame(X, columns=cols)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("Entrenamiento:", Xtr.shape, " Prueba:", Xte.shape, " Proporción clase 1:", round(y.mean(), 3))


## Parte 1 — Un árbol solo vs. bagging  *(presentación B1–B4)*

Primero un único árbol sin podar (varianza alta, como vimos anteriormente), y después un `BaggingClassifier` con 200 árboles del mismo tipo.

In [ ]:
arbol_solo = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
cv_arbol = cross_val_score(DecisionTreeClassifier(random_state=0), Xtr, ytr, cv=kf).mean()

bagging = ...     # ✏️ BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=200, n_jobs=-1, random_state=0).fit(Xtr, ytr)
cv_bagging = cross_val_score(BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=200, n_jobs=-1, random_state=0), Xtr, ytr, cv=kf).mean()

print(f"Árbol solo      : exactitud CV = {cv_arbol:.3f}")
print(f"Bagging (200)   : exactitud CV = {cv_bagging:.3f}")


✏️ **Pregunta 1.1.** (a) ¿Cuál de los dos tiene mejor exactitud de validación cruzada? (b) ¿Por qué un solo árbol sin podar tiene varianza alta? (c) ¿Qué le pasaría a la exactitud del bagging si usaras solo 5 árboles en vez de 200?

**Respuesta:**

## Parte 2 — Random Forest: el ingrediente extra  *(presentación C1–C2)*

Comparamos qué tan seguido una sola variable domina la raíz de los árboles, con y sin restringir `max_features`, para ver si Random Forest realmente diversifica los árboles.

In [ ]:
def raices(modelo_fn, n_reps=30):
    conteo = np.zeros(X.shape[1], dtype=int)
    for i in range(n_reps):
        idx = np.random.RandomState(i).choice(len(Xtr), len(Xtr), replace=True)
        arbol = modelo_fn(i).fit(Xtr.values[idx], ytr[idx] if hasattr(ytr, "__getitem__") else np.asarray(ytr)[idx])
        conteo[arbol.tree_.feature[0]] += 1
    return conteo

conteo_sin_restriccion = raices(lambda s: DecisionTreeClassifier(max_depth=1, random_state=s))
conteo_rf = ...     # ✏️ raices(lambda s: DecisionTreeClassifier(max_depth=1, max_features="sqrt", random_state=s))

print("Variable más usada en la raíz (sin restringir):", cols[conteo_sin_restriccion.argmax()], conteo_sin_restriccion.max(), "/30")
print("Variable más usada en la raíz (max_features='sqrt'):", cols[conteo_rf.argmax()], conteo_rf.max(), "/30")


✏️ **Pregunta 2.1.** ¿Cambió cuál variable domina la raíz, o solo cuántas veces domina? ¿Qué nos dice esto sobre la diversidad de los árboles en cada caso?

**Respuesta:**

## Parte 3 — Error OOB e importancias  *(presentación C3–C5)*

Entrenamos un `RandomForestClassifier` con `oob_score=True` y graficamos cómo se estabiliza el error OOB al aumentar `n_estimators`. También extraemos sus importancias de variable.

In [ ]:
n_list = [5, 10, 20, 40, 80, 150, 300]
oob_err = []
for ne in n_list:
    rf = RandomForestClassifier(n_estimators=ne, oob_score=True, max_features="sqrt",
                                 random_state=0, n_jobs=-1).fit(Xtr, ytr)
    oob_err.append(1 - rf.oob_score_)

plt.figure(figsize=(5.5, 3))
plt.plot(n_list, oob_err, "o-")
plt.xlabel("n_estimators"); plt.ylabel("error OOB"); plt.title("Error OOB vs número de árboles")
plt.grid(alpha=0.3); plt.show()

bosque_final = ...     # ✏️ RandomForestClassifier(n_estimators=300, max_features="sqrt", oob_score=True, random_state=0, n_jobs=-1).fit(Xtr, ytr)
imp = pd.Series(bosque_final.feature_importances_, index=cols).sort_values(ascending=False)
print(imp)


✏️ **Pregunta 3.1.** (a) ¿En qué número de árboles se estabiliza el error OOB? (b) Según `feature_importances_`, ¿qué variables parecen ser ruido (las de menor importancia)? (c) ¿El error de prueba final de `bosque_final` se parece a su error OOB? Verifícalo con código.

**Respuesta:**

## Parte 4 — Comparación final: árbol solo, bagging y Random Forest  *(presentación C7)*

Reunimos los tres modelos en una tabla con exactitud de prueba y tiempo de entrenamiento.

In [ ]:
modelos = {
    "Árbol solo": DecisionTreeClassifier(random_state=0),
    "Bagging": BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=200, n_jobs=-1, random_state=0),
    "Random Forest": bosque_final,
}
filas = []
for nombre, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(Xtr, ytr)
    dt = time.time() - t0
    filas.append({"modelo": nombre, "exactitud_prueba": round(modelo.score(Xte, yte), 3), "tiempo_entrenamiento_s": round(dt, 3)})
tabla_final = pd.DataFrame(filas).sort_values("exactitud_prueba", ascending=False)
tabla_final.to_csv("entregable_unidad13.csv", index=False)
tabla_final


✏️ **Pregunta 4.1.** (4 a 6 líneas) Con la tabla anterior: (1) ¿qué modelo elegirías para este problema y por qué? (2) ¿Qué modelo tiene el mejor balance entre exactitud y tiempo? (3) Según C7 de la presentación, ¿qué se pierde al pasar de un árbol solo a un Random Forest, aunque se gane estabilidad?

**Respuesta:**